# MBD Prior Validation — ⟨h|h⟩ Inner Product Comparison

Draws random samples from the prior and checks that the MBD waveform gives the same
`⟨h|h⟩` inner product as the UFD waveform to within tolerance.

**Prior ranges:**
- Chirp mass: 5×10⁵ – 1×10⁶ M☉
- Mass ratio q: 0.25 – 1.0
- χ₁z, χ₂z: 0.0 – 0.99
- Luminosity distance: 1.5×10⁵ – 5×10⁵ Mpc

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings

from dingo.gw.domains import build_domain
from dingo.gw.domains.multibanded_frequency_domain import MultibandedFrequencyDomain
from dingo.gw.waveform_generator.waveform_generator import BBHxWaveformGenerator

def to_np(x):
    """Convert CuPy or numpy array to numpy."""
    return x.get() if hasattr(x, "get") else np.asarray(x)

def chirp_q_to_m1m2(chirp_mass, q):
    """Convert (chirp mass, q=m2/m1≤1) to (m1, m2) in solar masses."""
    eta = q / (1.0 + q) ** 2
    m_total = chirp_mass / eta ** (3.0 / 5.0)
    m1 = m_total / (1.0 + q)
    m2 = q * m1
    return m1, m2

print("Imports OK")

In [ ]:
# ── Domain settings ───────────────────────────────────────────────────────────
BASE_DOMAIN_SETTINGS = dict(
    type='UniformFrequencyDomain', f_min=1e-4, f_max=2e-2, delta_f=1e-7,
)
base_domain = build_domain(BASE_DOMAIN_SETTINGS)

MBD_NODES        = [1.0e-4, 1.0e-3, 4.0e-3, 1.0e-2]
MBD_DELTA_F_INIT = 1e-7
mbd = MultibandedFrequencyDomain(
    nodes=MBD_NODES, delta_f_initial=MBD_DELTA_F_INIT,
    base_domain=BASE_DOMAIN_SETTINGS,
)

print(f"UFD bins : {len(base_domain):,}")
print(f"MBD bins : {len(mbd):,}")
print(f"MBD bands: {mbd.num_bands}  (max decimation {mbd._decimation_factors_bands[-1]}×)")

In [ ]:
# ── Generator settings ────────────────────────────────────────────────────────
GEN_KWARGS = dict(
    approximant='PhenomHM', f_ref=0., use_gpu=True,
    direct_response=True,
    bbhx_t_obs_start_years=0.0, bbhx_t_obs_end_years=2,
    default_t_ref_years=0.75, bbhx_length=1024,
    isco_cutoff=True, isco_taper_width=0.1,
)

gen_base = BBHxWaveformGenerator(domain=base_domain, **GEN_KWARGS)
gen_mbd  = BBHxWaveformGenerator(domain=mbd,         **GEN_KWARGS)

print("Generators ready")
print(f"  BBHx sparse grid size : {gen_base.bbhx_length} pts  (UFD)")
print(f"  BBHx sparse grid size : {gen_mbd.bbhx_length}  pts  (MBD — same, interp to MBD after)")

In [ ]:
# ── Prior ranges ──────────────────────────────────────────────────────────────
PRIOR = dict(
    chirp_mass = (5e5,  1e6),       # solar masses
    q          = (0.25, 1.0),       # mass ratio m2/m1
    chi_1z     = (0.0,  0.99),
    chi_2z     = (0.0,  0.99),
    dist       = (1.5e5, 5e5),      # Mpc
    # extrinsic (randomised for completeness)
    theta_jn   = (0.0,  np.pi),
    phase      = (0.0,  2 * np.pi),
    ra         = (0.0,  2 * np.pi),
    dec        = (-np.pi / 2, np.pi / 2),
    psi        = (0.0,  np.pi),
)

N_SAMPLES  = 200   # number of prior draws
REL_TOL    = 1e-3  # flag samples above this threshold
RNG        = np.random.default_rng(42)

def sample_prior(n=1):
    """Draw n independent uniform samples from PRIOR; return list of param dicts."""
    samples = []
    for _ in range(n):
        p = {k: float(RNG.uniform(*v)) for k, v in PRIOR.items()}
        m1, m2 = chirp_q_to_m1m2(p.pop("chirp_mass"), p.pop("q"))
        p["mass_1"] = m1
        p["mass_2"] = m2
        p["geocent_time"] = 0.0   # fixed reference time
        samples.append(p)
    return samples

# quick sanity check
s = sample_prior(3)
print("Example sample:")
for k, v in s[0].items():
    print(f"  {k:20s} = {v:.4g}")

In [ ]:
# ── Validation loop ───────────────────────────────────────────────────────────
df_base = float(base_domain.delta_f)
df_mbd  = to_np(mbd.delta_f)           # per-bin delta_f array, shape (n_mbd,)

results = []   # list of dicts: {params, inner_base, inner_mbd, rel_err, failed}

samples = sample_prior(N_SAMPLES)

for i, params in enumerate(samples):
    row = {"params": params, "failed": False, "fail_reason": ""}

    try:
        wf_base = gen_base.generate_amp_phase(params, catch_waveform_errors=False)
        wf_mbd  = gen_mbd.generate_amp_phase(params,  catch_waveform_errors=False)
    except Exception as exc:
        row["failed"] = True
        row["fail_reason"] = str(exc)
        results.append(row)
        print(f"[{i+1:3d}/{N_SAMPLES}] FAIL  m1={params['mass_1']:.3e}  m2={params['mass_2']:.3e}  {exc}")
        continue

    # waveform shape: (1, 3, n_freqs) or (3, n_freqs) — take channel A
    h_base_A = to_np(wf_base["waveform"]).reshape(-1, wf_base["waveform"].shape[-1] if hasattr(wf_base["waveform"], "shape") else len(to_np(wf_base["waveform"]).ravel()))[0]
    h_mbd_A  = to_np(wf_mbd["waveform"]).reshape(-1,  wf_mbd["waveform"].shape[-1]  if hasattr(wf_mbd["waveform"],  "shape") else len(to_np(wf_mbd["waveform"]).ravel()))[0]

    # Robust shape handling: squeeze batch dim, pick first channel
    def get_channel_A(wf_dict):
        arr = to_np(wf_dict["waveform"])
        # possible shapes: (n_freqs,), (3, n_freqs), (1, 3, n_freqs)
        while arr.ndim > 1 and arr.shape[0] in (1, 3):
            arr = arr[0]
        return arr  # shape (n_freqs,)

    h_base_A = get_channel_A(wf_base)
    h_mbd_A  = get_channel_A(wf_mbd)

    inner_base = float(np.sum(np.abs(h_base_A) ** 2) * df_base)
    inner_mbd  = float(np.sum(np.abs(h_mbd_A)  ** 2 * df_mbd))

    if inner_base == 0.0:
        row["failed"] = True
        row["fail_reason"] = "zero UFD inner product"
        results.append(row)
        print(f"[{i+1:3d}/{N_SAMPLES}] ZERO  m1={params['mass_1']:.3e}  m2={params['mass_2']:.3e}")
        continue

    rel_err = abs(inner_mbd - inner_base) / inner_base
    row.update({"inner_base": inner_base, "inner_mbd": inner_mbd, "rel_err": rel_err})
    results.append(row)

    flag = " *** ABOVE TOL ***" if rel_err > REL_TOL else ""
    if (i + 1) % 20 == 0 or rel_err > REL_TOL:
        print(f"[{i+1:3d}/{N_SAMPLES}]  rel_err={rel_err:.2e}  "
              f"m1={params['mass_1']:.3e}  chi1={params['chi_1z']:.2f}{flag}")

print("\nDone.")

In [ ]:
# ── Summary statistics ────────────────────────────────────────────────────────
good    = [r for r in results if not r["failed"] and "rel_err" in r]
failed  = [r for r in results if r["failed"]]
flagged = [r for r in good if r["rel_err"] > REL_TOL]

errs = np.array([r["rel_err"] for r in good])

print(f"Total samples   : {N_SAMPLES}")
print(f"Successful      : {len(good)}")
print(f"Failed          : {len(failed)}")
print(f"Above tol ({REL_TOL:.0e}) : {len(flagged)}")
print()
if len(good):
    print(f"Relative ⟨h|h⟩ error statistics:")
    print(f"  min    : {errs.min():.3e}")
    print(f"  median : {np.median(errs):.3e}")
    print(f"  p95    : {np.percentile(errs, 95):.3e}")
    print(f"  max    : {errs.max():.3e}")
    print()
    idx_worst = int(np.argmax(errs))
    wp = good[idx_worst]["params"]
    print(f"Worst sample (rel_err={errs[idx_worst]:.3e}):")
    for k in ["mass_1", "mass_2", "chi_1z", "chi_2z", "dist"]:
        print(f"  {k:12s} = {wp[k]:.4g}")

if failed:
    print(f"\nFailed samples:")
    for r in failed:
        p = r["params"]
        print(f"  m1={p['mass_1']:.3e}  m2={p['mass_2']:.3e}  reason: {r['fail_reason'][:80]}")

In [ ]:
# ── Plots ─────────────────────────────────────────────────────────────────────
if len(good) == 0:
    print("No successful samples to plot.")
else:
    m1s  = np.array([r["params"]["mass_1"] for r in good])
    m2s  = np.array([r["params"]["mass_2"] for r in good])
    chi1 = np.array([r["params"]["chi_1z"] for r in good])
    chi2 = np.array([r["params"]["chi_2z"] for r in good])
    mchirp = (m1s * m2s) ** (3/5) / (m1s + m2s) ** (1/5)
    q_arr  = m2s / m1s

    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    fig.suptitle(f"⟨h|h⟩ Relative Error — {len(good)} prior samples  (tol={REL_TOL:.0e})", fontsize=13)

    scatter_kw = dict(c=errs, cmap="plasma", norm=plt.matplotlib.colors.LogNorm(
        vmin=max(errs.min(), 1e-8), vmax=max(errs.max(), REL_TOL * 2)), s=18, alpha=0.8)

    def labeled_scatter(ax, x, y, xlabel, ylabel):
        sc = ax.scatter(x, y, **scatter_kw)
        ax.axhline(REL_TOL, color="red", lw=1, ls="--", label=f"tol={REL_TOL:.0e}")
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)
        return sc

    sc = labeled_scatter(axes[0, 0], mchirp / 1e6, errs, "Chirp mass [10⁶ M☉]", "Rel. error")
    labeled_scatter(axes[0, 1], q_arr,  errs, "Mass ratio q", "Rel. error")
    labeled_scatter(axes[0, 2], chi1,   errs, "χ₁z", "Rel. error")
    labeled_scatter(axes[1, 0], chi2,   errs, "χ₂z", "Rel. error")
    labeled_scatter(axes[1, 1], chi1 + chi2, errs, "χ₁z + χ₂z", "Rel. error")

    # Error distribution histogram
    ax = axes[1, 2]
    ax.hist(np.log10(errs + 1e-20), bins=30, color="steelblue", edgecolor="white", lw=0.4)
    ax.axvline(np.log10(REL_TOL), color="red", lw=1.5, ls="--", label=f"tol={REL_TOL:.0e}")
    ax.set_xlabel("log₁₀(rel. error)")
    ax.set_ylabel("Count")
    ax.legend(fontsize=9)

    for ax in axes.flat[:5]:
        ax.set_yscale("log")
        ax.legend(fontsize=8)

    plt.colorbar(sc, ax=axes[0, :], label="Rel. ⟨h|h⟩ error", shrink=0.8, pad=0.02)
    plt.tight_layout()
    plt.show()

    # Pass/fail summary line
    pct_pass = 100 * (1 - len(flagged) / len(good))
    print(f"\n{pct_pass:.1f}% of samples pass (rel_err < {REL_TOL:.0e})")